In [2]:
# ============================================================
# MACHINE LEARNING PIPELINE — WINE (CLASSIFICATION) + CARS (REGRESSION)
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_squared_error, r2_score

# ============================================================
# 1. LOAD & EXPLORE WINE DATASETS
# ============================================================

white = pd.read_csv("winequality-white.csv", sep=";")
red = pd.read_csv("winequality-red.csv", sep=";")

wine = pd.concat([white, red], axis=0)

print("\nWine Dataset Shape:", wine.shape)
print("\nFirst 5 rows:")
print(wine.head())

# Features & target
X_wine = wine.drop("quality", axis=1)
y_wine = wine["quality"]

# ============================================================
# 2. SPLIT WINE DATA
# ============================================================

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42
)

# Scale features
scaler_w = StandardScaler()
X_train_w_scaled = scaler_w.fit_transform(X_train_w)
X_test_w_scaled = scaler_w.transform(X_test_w)

# ============================================================
# 3. CLASSIFICATION MODELS
# ============================================================

results_classification = {}

# Logistic Regression
log_reg = LogisticRegression(max_iter=300)
log_reg.fit(X_train_w_scaled, y_train_w)
pred_lr = log_reg.predict(X_test_w_scaled)
results_classification["Logistic Regression"] = accuracy_score(y_test_w, pred_lr)

# SVC
svc = SVC(C=1.0, kernel="rbf")
svc.fit(X_train_w_scaled, y_train_w)
pred_svc = svc.predict(X_test_w_scaled)
results_classification["SVC"] = accuracy_score(y_test_w, pred_svc)

# KNN Classifier
knn_c = KNeighborsClassifier(n_neighbors=5)
knn_c.fit(X_train_w_scaled, y_train_w)
pred_knn_c = knn_c.predict(X_test_w_scaled)
results_classification["KNN Classifier"] = accuracy_score(y_test_w, pred_knn_c)

print("\n==============================")
print("CLASSIFICATION RESULTS")
print("==============================")
for model, acc in results_classification.items():
    print(f"{model}: {acc:.4f}")

print("\nClassification Report (SVC):")
print(classification_report(y_test_w, pred_svc, zero_division=0))

# ============================================================
# 4. LOAD & CLEAN CAR DATASET
# ============================================================

cars = pd.read_csv("used_cars.csv")

print("\nCar Dataset (first 5 rows):")
print(cars.head())

# --- CLEAN PRICE COLUMN ---
cars["price"] = cars["price"].str.replace("$", "", regex=False)
cars["price"] = cars["price"].str.replace(",", "", regex=False)
cars["price"] = pd.to_numeric(cars["price"], errors="coerce")

# --- CLEAN MILEAGE COLUMN ---
cars["milage"] = cars["milage"].str.replace("mi.", "", regex=False)
cars["milage"] = cars["milage"].str.replace(",", "", regex=False)
cars["milage"] = pd.to_numeric(cars["milage"], errors="coerce")

# Drop rows with missing numeric values
cars = cars.dropna(subset=["price", "milage"])

# Select numeric features only
numeric_cols = cars.select_dtypes(include=[np.number]).columns
cars = cars[numeric_cols]

X_car = cars.drop("price", axis=1)
y_car = cars["price"]

# ============================================================
# 5. SPLIT CAR DATA
# ============================================================

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_car, y_car, test_size=0.2, random_state=42
)

# Scale
scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
X_test_c_scaled = scaler_c.transform(X_test_c)

# ============================================================
# 6. REGRESSION MODELS
# ============================================================

results_regression = {}

# Linear Regression
lin = LinearRegression()
lin.fit(X_train_c_scaled, y_train_c)
pred_lin = lin.predict(X_test_c_scaled)
results_regression["Linear Regression (R2)"] = r2_score(y_test_c, pred_lin)

# SVR
svr = SVR(C=1.0, kernel="rbf")
svr.fit(X_train_c_scaled, y_train_c)
pred_svr = svr.predict(X_test_c_scaled)
results_regression["SVR (R2)"] = r2_score(y_test_c, pred_svr)

# KNN Regressor
knn_r = KNeighborsRegressor(n_neighbors=5)
knn_r.fit(X_train_c_scaled, y_train_c)
pred_knn_r = knn_r.predict(X_test_c_scaled)
results_regression["KNN Regressor (R2)"] = r2_score(y_test_c, pred_knn_r)

print("\n==============================")
print("REGRESSION RESULTS (R² Score)")
print("==============================")
for model, score in results_regression.items():
    print(f"{model}: {score:.4f}")

# ============================================================
# 7. HYPERPARAMETER TUNING (CLASSIFICATION)
# ============================================================

print("\n==============================")
print("HYPERPARAMETER TUNING — CLASSIFICATION")
print("==============================")

# Tune SVC C parameter
for C in [0.1, 1, 10]:
    svc_tuned = SVC(C=C)
    svc_tuned.fit(X_train_w_scaled, y_train_w)
    pred = svc_tuned.predict(X_test_w_scaled)
    print(f"SVC (C={C}) Accuracy: {accuracy_score(y_test_w, pred):.4f}")

# Tune KNN k parameter
for k in [3, 5, 11]:
    knn_tuned = KNeighborsClassifier(n_neighbors=k)
    knn_tuned.fit(X_train_w_scaled, y_train_w)
    pred = knn_tuned.predict(X_test_w_scaled)
    print(f"KNN (k={k}) Accuracy: {accuracy_score(y_test_w, pred):.4f}")

# Tune Logistic Regression regularization
for C in [0.1, 1, 10]:
    lr_tuned = LogisticRegression(C=C, max_iter=300)
    lr_tuned.fit(X_train_w_scaled, y_train_w)
    pred = lr_tuned.predict(X_test_w_scaled)
    print(f"Logistic Regression (C={C}) Accuracy: {accuracy_score(y_test_w, pred):.4f}")


Wine Dataset Shape: (6497, 12)

First 5 rows:
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.0              0.27         0.36            20.7      0.045   
1            6.3              0.30         0.34             1.6      0.049   
2            8.1              0.28         0.40             6.9      0.050   
3            7.2              0.23         0.32             8.5      0.058   
4            7.2              0.23         0.32             8.5      0.058   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 45.0                 170.0   1.0010  3.00       0.45   
1                 14.0                 132.0   0.9940  3.30       0.49   
2                 30.0                  97.0   0.9951  3.26       0.44   
3                 47.0                 186.0   0.9956  3.19       0.40   
4                 47.0                 186.0   0.9956  3.19       0.40   

   alcohol  quality  
0      8.8       